In [ ]:
import cv2
import numpy as np
import joblib
import dlib
from skimage import feature

In [ ]:
# Carga de modelos
SVM_PATH = "models/svm_lbp_barba.joblib"
SCALER_PATH = "models/scaler_lbp_barba.joblib"

try:
    svm_lbp = joblib.load(SVM_PATH)
    scaler_lbp = joblib.load(SCALER_PATH)
except FileNotFoundError:
    print(f"No se encuentran los modelos en {SVM_PATH} o {SCALER_PATH}")
    exit()

WIDTH, HEIGHT = 192, 192
CLASS_LABELS = ["con_barba", "sin_barba"]

In [ ]:
# Configuración de dlib
detector_dlib = dlib.get_frontal_face_detector()
LANDMARKS_PATH = "shape_predictor_68_face_landmarks.dat"
try:
    predictor_dlib = dlib.shape_predictor(LANDMARKS_PATH)
except RuntimeError:
    print(f"[ERROR] Falta '{LANDMARKS_PATH}'.")
    exit()

In [ ]:
# FUNCIONES DE SOPORTE
def lbphist(gray, ncellsx, ncellsy, width, height, lbp_method):
    pxpercellx = int(width / ncellsx)
    pxpercelly = int(height / ncellsy)
    ofx = int((width - int(pxpercellx) * ncellsx) / 2)
    ofy = int((height - int(pxpercelly) * ncellsy) / 2)
    LBPu_hist = []
    for i in range(ncellsy):
        for j in range(ncellsx):
            roi = gray[ofy + i * pxpercelly:ofy + (i + 1) * pxpercelly,
                       ofx + j * pxpercellx:ofx + (j + 1) * pxpercellx]
            lbpimg = feature.local_binary_pattern(roi, 8, 1, method=lbp_method)
            n_bins = 256
            feath, _ = np.histogram(lbpimg, density=False, bins=n_bins, range=(0, n_bins))
            LBPu_hist = np.concatenate([LBPu_hist, feath])
    return LBPu_hist

def preprocess_lbp_from_gray(gray):
    gray_resized = cv2.resize(gray, (WIDTH, HEIGHT), interpolation=cv2.INTER_AREA)
    feat_lbp = lbphist(gray_resized, ncellsx=3, ncellsy=3, width=WIDTH, height=HEIGHT, lbp_method="nri_uniform")
    desc = feat_lbp.astype("float32").reshape(1, -1)
    desc_scaled = scaler_lbp.transform(desc)
    return desc_scaled

def overlay_image_alpha(img, img_overlay, x, y, w_overlay, h_overlay):
    if img_overlay is None: return img
    h, w, _ = img.shape
    
    try:
        img_overlay = cv2.resize(img_overlay, (w_overlay, h_overlay))
    except: return img

    y1, y2 = max(0, y), min(h, y + h_overlay)
    x1, x2 = max(0, x), min(w, x + w_overlay)

    if y1 >= y2 or x1 >= x2: return img

    y_overlay_start = max(0, -y)
    x_overlay_start = max(0, -x)
    y_overlay_end = y_overlay_start + (y2 - y1)
    x_overlay_end = x_overlay_start + (x2 - x1)

    overlay_crop = img_overlay[y_overlay_start:y_overlay_end, x_overlay_start:x_overlay_end]
    background_crop = img[y1:y2, x1:x2]

    alpha_mask = overlay_crop[:, :, 3] / 255.0
    alpha_inv = 1.0 - alpha_mask

    for c in range(0, 3):
        background_crop[:, :, c] = (alpha_mask * overlay_crop[:, :, c] + alpha_inv * background_crop[:, :, c])

    img[y1:y2, x1:x2] = background_crop
    return img

def get_largest_face_dlib(frame):
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    rects = detector_dlib(gray, 0)
    if len(rects) == 0: return None, None, None
    largest_rect = max(rects, key=lambda r: r.width() * r.height())
    shape_dlib = predictor_dlib(gray, largest_rect)
    shape = np.array([[p.x, p.y] for p in shape_dlib.parts()])
    x, y, w, h = largest_rect.left(), largest_rect.top(), largest_rect.width(), largest_rect.height()
    return (x, y, w, h), shape

[INFO] Modelos de barba cargados correctamente.
Iniciando Filtro Final. Pulsa ESC para salir.


In [ ]:

def main():
    img_saiyan = cv2.imread("filter_assets/super_saiyan_hair.png", -1)
    img_kawaii_hair = cv2.imread("filter_assets/anime_girl_hair.png", -1)
    img_kawaii_eyes = cv2.imread("filter_assets/anime_girl_eyes.png", -1)

    cap = cv2.VideoCapture(0)
    print("Iniciando Filtro Final. Pulsa ESC para salir.")

    while True:
        ret, frame = cap.read()
        if not ret: break

        (rect, shape) = get_largest_face_dlib(frame)[0:2]

        if rect is not None and shape is not None:
            x, y, w, h = rect
            
            # PREDICCIÓN
            offset = int(w * 0.1)
            y1, y2 = max(0, y - offset), min(frame.shape[0], y + h + offset)
            x1, x2 = max(0, x - offset), min(frame.shape[1], x + w + offset)
            face_roi = frame[y1:y2, x1:x2]

            label = "sin_barba"
            if face_roi.size > 0:
                try:
                    gray_roi = cv2.cvtColor(face_roi, cv2.COLOR_BGR2GRAY)
                    desc = preprocess_lbp_from_gray(gray_roi)
                    pred_idx = int(svm_lbp.predict(desc)[0])
                    label = CLASS_LABELS[pred_idx]
                except: pass

            bridge_nose = shape[27] # Punto entre ojos
            face_width_pts = np.linalg.norm(shape[16] - shape[0]) 
            eyebrow_level = min(shape[19][1], shape[24][1])
            
            text_to_show = ""
            text_color = (255, 255, 255)

            if label == "con_barba":
                # MODO SAIYAN
                text_to_show = "SUPER SAIYAN MODE"
                text_color = (0, 255, 255) # Amarillo

                hair_w = int(face_width_pts * 2.0)
                hair_h = int(hair_w * 0.9)
                hair_x = bridge_nose[0] - (hair_w // 2)
                hair_y = eyebrow_level - int(hair_h * 0.90) 

                overlay_image_alpha(frame, img_saiyan, hair_x, hair_y, hair_w, hair_h)
            
            else:
                # MODO KAWAII
                text_to_show = "KAWAII MODE"
                text_color = (255, 180, 255) # Rosa

                hair_w = int(face_width_pts * 1.6)
                hair_h = int(hair_w * 1.3)
                hair_x = bridge_nose[0] - (hair_w // 2)
                hair_y = eyebrow_level - int(hair_h * 0.25)
                
                overlay_image_alpha(frame, img_kawaii_hair, hair_x, hair_y, hair_w, hair_h)

                if img_kawaii_eyes is not None:
                    # Punto 36 ojo izquierdo y 45 ojo derecho
                    eyes_span_width = np.linalg.norm(shape[45] - shape[36]) # distancia total que cubren los ojos
                    
                    overlay_eyes_w = int(eyes_span_width * 1.3)
                    overlay_eyes_h = int(overlay_eyes_w * 0.4)

                    # Centrado en el punto de la nariz (Punto 27)
                    eyes_x = bridge_nose[0] - (overlay_eyes_w // 2)
                    eyes_y = bridge_nose[1] - (overlay_eyes_h // 2)

                    overlay_image_alpha(frame, img_kawaii_eyes, eyes_x, eyes_y, overlay_eyes_w, overlay_eyes_h)

            # Posición del texto debajo de la cara
            cv2.rectangle(frame, (x, y), (x+w, y+h), text_color, 2)
            cv2.putText(frame, text_to_show, (x, y + h + 40), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, text_color, 2, cv2.LINE_AA)

        cv2.imshow("Filtro VC Final", frame)
        if cv2.waitKey(1) & 0xFF == 27: break

    cap.release()
    cv2.destroyAllWindows()

if __name__ == "__main__":
    main()